<strong><span style="color:Orange;font-size:40px">News Summarisation language model</strong><br>
<strong><span style="color:darkorange;font-size:20px"> Objectives</strong><br>
- develop a language model able to injest raw text format reviews, classify and summarise the review<br>
- Processing of raw input to classification of subject, sentiment ranking<br>
- recomend the next steps to rectify if a complaint

<span style="color:skyblue;font-size:20px"> Imports

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline, BartTokenizer, BartForConditionalGeneration
import spacy
import torch
import evaluate
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.8"

device = "mps" if torch.backends.mps.is_available() else "cpu"


developmental_run = True

<span style="color:skyblue;font-size:20px"> Raw Data collection, Train, validate and Test

In [9]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", trust_remote_code=True)

# 🔽 Add this directly after loading
dataset["train"] = dataset["train"].select(range(2000))
dataset["validation"] = dataset["validation"].select(range(500))
dataset["test"] = dataset["test"].select(range(200))

print(dataset)

print(len(dataset["train"]))

# display the raw data structure
# print(dataset)
# display(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 200
    })
})
2000


<span style="font-size:20px;color:orange"> Full/ Dev run

In [10]:
train_data = dataset["train"].to_pandas()

if developmental_run:
    train_data = train_data.head(20000)
    print(f"Dev Run: {train_data.shape}")
else:
    print(f"Full Run: {train_data.shape}")

print(train_data.columns)

Dev Run: (2000, 3)
Index(['article', 'highlights', 'id'], dtype='object')


<span style="color:skyblue;font-size:20px">Tokenisation ready for the transformer

<span style="font-size:20px;color:skyblue">Initiation of transformer - untrained

In [11]:
# summeriser = pipeline(
#     "summarization",
#     model="google/long-t5-tglobal-base",
#     device=device,
# )

# tokeniser = BartTokenizer.from_pretrained("facebook/bart-large")

# text = train_data['article'].tolist()
# # text = tokeniser
# summaries = []
# summeries = summeriser(
#     text,
#     # min_length=30,
#     do_sample=False,
# )
# for summary in summeries:
#     summaries.append(summary['summary_text'])
# train_data['summary'] = summaries
# train_data = train_data.rename(columns={'highlights': 'target'})

In [12]:
# display(train_data.head())

In [13]:
# rouge = evaluate.load("rouge")
# results = rouge.compute(
#     predictions=train_data['summary'].tolist(),
#     references=train_data['target'].tolist(),
#     use_stemmer=True,
# )
# print(f"ROUGE-1: {results['rouge1']}")

<span style="font-size:20px;color:skyblue">Training regieme for fine tuning

In [18]:
import os
import torch
import gc
import numpy as np
import pandas as pd
from itertools import product
from tqdm import tqdm
from rouge_score import rouge_scorer
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import load_from_disk

os.environ["PYTORCH_MPS_LOW_WATERMARK_RATIO"] = "0.0"

fresh_tokenise = False

# Select Device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🖥️ Using device: {device}")

# Tokenizer + Preprocess
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_batch(batch):
    inputs = tokenizer(batch['article'], max_length=256, truncation=True, padding="max_length")
    targets = tokenizer(batch['highlights'], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs



if fresh_tokenise:
    print("🔄 Tokenising dataset...")
    tokenised_ds = dataset.map(preprocess_batch, batched=True)
    print("tokeniseation of data complete")
    print("saving tokenised dataset to disk...")
    tokenised_ds.save_to_disk("tokenised_cnn_dailymail")
    print("Tokenised dataset saved.")
else:
    print("🔄 Loading existing tokenised dataset...")
    tokenised_ds = load_from_disk("tokenised_cnn_dailymail")
    print(f"Loaded tokenised dataset with {len(tokenised_ds['train'])} training examples.")

# Define ROUGE evaluation function
def compute_rouge(preds, refs):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = [scorer.score(p, r) for p, r in zip(preds, refs)]
    return {
        metric: sum(s[metric].fmeasure for s in scores) / len(scores)
        for metric in ['rouge1', 'rouge2', 'rougeL']
    }

# === Hyperparameter Search Space ===
param_grid = {
    "learning_rate": [3e-5],
    "per_device_train_batch_size": [3, 5],
    "num_train_epochs": [1 ,3, 5],
    "weight_decay": [0.1],
}

grid_combinations = list(product(*param_grid.values()))
best_score = 0.0
best_params = {}

# === Loop over Hyperparams ===
for values in grid_combinations:
    params = dict(zip(param_grid.keys(), values))
    run_id = f"lr{params['learning_rate']}_bs{params['per_device_train_batch_size']}_ep{params['num_train_epochs']}_wd{params['weight_decay']}"
    output_dir = f"models/{run_id}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n🔧 Trying config: {params}")

    # If already trained, load model
    if os.path.exists(os.path.join(output_dir, "pytorch_model.bin")):
        print("📦 Loading existing model...")
        model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(device)
    else:
        print("🚂 Training model...")
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

        args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=params["learning_rate"],
            per_device_train_batch_size=params["per_device_train_batch_size"],
            num_train_epochs=params["num_train_epochs"],
            weight_decay=params["weight_decay"],
            logging_steps=50,
            load_best_model_at_end=True,
            save_total_limit=1,
            report_to="none",
            push_to_hub=False,
        )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenised_ds["train"],
        eval_dataset=tokenised_ds["validation"],
        tokenizer=tokenizer,
    )

    trainer.train()

    # ✅ Save model and tokenizer
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("💾 Model saved.")

    # ✅ Reload the model and tokenizer to ensure completeness
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(device)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print("🔄 Model reloaded from saved checkpoint.")

    # === Evaluate on test set ===
    print("🔍 Evaluating on test set...")
    test_articles = tokenised_ds["test"]["article"][:10]
    test_refs = tokenised_ds["test"]["highlights"][:10]
    preds = []

    for article in tqdm(test_articles):
        inputs = tokenizer(article, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=256, num_beams=4)
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        preds.append(decoded)

        # Clean up
        del inputs, outputs
        if device == "mps":
            torch.mps.empty_cache()
        gc.collect()

    rouge = compute_rouge(preds, test_refs)
    score = rouge["rougeL"]
    print(f"📊 ROUGE-L: {score:.4f}")

    if score > best_score:
        best_score = score
        best_params = params

# === Summary ===
print(f"\n🏁 Best hyperparameters: {best_params}")
print(f"📈 Best ROUGE-L: {best_score:.4f}")

🖥️ Using device: mps
🔄 Tokenising dataset...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

tokeniseation of data complete
saving tokenised dataset to disk...


Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,1.300300,0.948260
2,1.246400,0.932211
3,1.216700,0.930421
4,1.180000,0.929195
5,1.192300,0.928869


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pin

💾 Model saved.
🔄 Model reloaded from saved checkpoint.
🔍 Evaluating on test set...


100%|██████████| 10/10 [00:17<00:00,  1.77s/it]

📊 ROUGE-L: 0.2196

🏁 Best hyperparameters: {'learning_rate': 3e-05, 'per_device_train_batch_size': 5, 'num_train_epochs': 5, 'weight_decay': 0.1}
📈 Best ROUGE-L: 0.2196


In [ ]:
print(f"\n🏁 Best Params: {best_params} with ROUGE-L Score: {best_score:.4f}")


🏁 Best Params: {'learning_rate': 3e-05, 'per_device_train_batch_size': 5, 'num_train_epochs': 5, 'weight_decay': 0.1} with ROUGE-L Score: 0.2524
